In [21]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
import json
import csv
from pathlib import Path
from collections import defaultdict
from sentence_transformers import SentenceTransformer
import numpy as np


# Install dependencies (run once)# !pip install sentence-transformers

In [36]:
# The two models to compare
BASE_MODEL = "gemma2:9b"        # Baseline explanations (ground truth)
COMPARE_MODEL = "internlm2:latest"    # Explanations to compare

BASE_TAG = BASE_MODEL.replace(":" , "_").replace(".", "_").replace("-", "_")
COMPARE_TAG = COMPARE_MODEL.replace(":", "_").replace(".", "_").replace("-", "_")

print(f"[INFO] Base model (baseline explanations): {BASE_MODEL} -> {BASE_TAG}")
print(f"[INFO] Compare model: {COMPARE_MODEL} -> {COMPARE_TAG}")

[INFO] Base model (baseline explanations): gemma2:9b -> gemma2_9b
[INFO] Compare model: internlm2:latest -> internlm2_latest


In [37]:
SA_DATASET_NAMES = ["DoS", "Fuzzy", "Gear", "RPM"]

SA_QUESTION_FILES = {
    "DoS":   Path("DoS_sa_qa/questions/dos_sa_questions.json"),
    "Fuzzy": Path("Fuzzy_sa_qa/questions/fuzzy_sa_questions.json"),
    "Gear":  Path("Gear_sa_qa/questions/gear_sa_questions.json"),
    "RPM":   Path("RPM_sa_qa/questions/rpm_sa_questions.json"),
}

def sa_answers_path(dataset_name: str, model_tag: str) -> Path:
    """Path to model short-answer file (primary source for llm_answer)."""
    return Path(f"{dataset_name}_sa_qa/llm_answers/{dataset_name.lower()}_sa_answers_{model_tag}.jsonl")


def sa_explanations_path(dataset_name: str, model_tag: str) -> Path:
    """Path to model explanation file (primary source for llm_explanation baseline/comparison)."""
    return Path(f"{dataset_name}_sa_qa/llm_explanations/{dataset_name.lower()}_sa_explanations_{model_tag}.jsonl")


def get_text_field(record: dict, preferred_keys: list) -> str:
    """Return first non-empty string value from candidate keys."""
    for key in preferred_keys:
        value = record.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
    return ""

In [38]:
def load_sa_answers(path: Path) -> list:
    """Load answer records from JSONL file."""
    records = []
    if not path.exists():
        return records
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records

def normalize_text(text: str) -> str:
    return " ".join(text.strip().split()).lower() if text else ""

In [39]:
# Load semantic similarity model
print("[INFO] Loading sentence transformer model...")
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")
print("[INFO] Model loaded successfully")

ANSWER_SIM_THRESHOLD = 0.90
EXPLANATION_SIM_THRESHOLD = 0.80


def compute_token_f1(pred: str, gold: str) -> float:
    """Token-level F1 score: overlap of tokens between prediction and gold."""
    if not pred and not gold:
        return 1.0
    if not pred or not gold:
        return 0.0

    pred_tokens = set(pred.split())
    gold_tokens = set(gold.split())

    if not pred_tokens or not gold_tokens:
        return 0.0

    intersection = len(pred_tokens & gold_tokens)
    precision = intersection / len(pred_tokens)
    recall = intersection / len(gold_tokens)

    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)


def compute_rouge_l(pred: str, gold: str) -> float:
    """ROUGE-L: Longest Common Subsequence based F1."""
    if not pred and not gold:
        return 1.0
    if not pred or not gold:
        return 0.0

    pred_tokens = pred.split()
    gold_tokens = gold.split()

    m, n = len(pred_tokens), len(gold_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if pred_tokens[i - 1] == gold_tokens[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    lcs_len = dp[m][n]
    precision = lcs_len / len(pred_tokens) if pred_tokens else 0.0
    recall = lcs_len / len(gold_tokens) if gold_tokens else 0.0

    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)


def compute_semantic_similarity(pred: str, gold: str) -> float:
    """Fast semantic similarity using sentence-transformers cosine similarity."""
    if not pred and not gold:
        return 1.0
    if not pred or not gold:
        return 0.0

    try:
        embeddings = semantic_model.encode([pred, gold], convert_to_numpy=True)
        similarity = np.dot(embeddings[0], embeddings[1]) / (
            np.linalg.norm(embeddings[0]) * np.linalg.norm(embeddings[1])
        )
        return float(similarity)
    except Exception as e:
        print(f"[WARN] Semantic similarity error: {e}")
        return 0.0

[INFO] Loading sentence transformer model...
[INFO] Model loaded successfully


In [40]:
def compute_comparison_records(base_explanations: list, compare_records: list) -> list:
    """
    Compare target LLM outputs using two tracks:
    1) llm_answer vs ground_truth (correctness at >= 90%)
    2) llm_explanation vs baseline llm_explanation (similarity + correctness at >= 80%)
    """
    base_map = {rec.get("qa_id"): rec for rec in base_explanations if rec.get("qa_id") is not None}

    eval_records = []

    for rec in compare_records:
        qa_id = rec.get("qa_id")
        if qa_id is None or qa_id not in base_map:
            continue

        base_rec = base_map[qa_id]

        answer_pred_raw = get_text_field(rec, ["llm_answer", "answer"])
        answer_gold_raw = get_text_field(rec, ["ground_truth", "answer"])
        if not answer_gold_raw:
            answer_gold_raw = get_text_field(base_rec, ["ground_truth", "answer"])

        answer_sim = compute_semantic_similarity(answer_pred_raw, answer_gold_raw)
        answer_is_correct = 1 if answer_sim >= ANSWER_SIM_THRESHOLD else 0

        explanation_pred_raw = get_text_field(rec, ["llm_explanation", "llm_answer", "explanation"])
        explanation_gold_raw = get_text_field(base_rec, ["llm_explanation", "llm_answer", "explanation"])

        expl_pred_norm = normalize_text(explanation_pred_raw)
        expl_gold_norm = normalize_text(explanation_gold_raw)

        expl_token_f1 = compute_token_f1(expl_pred_norm, expl_gold_norm)
        expl_rouge_l = compute_rouge_l(expl_pred_norm, expl_gold_norm)
        expl_semantic = compute_semantic_similarity(explanation_pred_raw, explanation_gold_raw)
        expl_is_correct = 1 if expl_semantic >= EXPLANATION_SIM_THRESHOLD else 0

        eval_records.append({
            "qa_id": qa_id,
            "sa_type": rec.get("sa_type", "unknown"),
            "answer_similarity": answer_sim,
            "answer_is_correct": answer_is_correct,
            "answer_pred": answer_pred_raw,
            "answer_gold": answer_gold_raw,
            "base_explanation": explanation_gold_raw,
            "compare_explanation": explanation_pred_raw,
            "expl_token_f1": expl_token_f1,
            "expl_rouge_l": expl_rouge_l,
            "expl_semantic_similarity": expl_semantic,
            "expl_is_correct": expl_is_correct,
        })

    return eval_records


def summary_from_records(records: list) -> dict:
    """Calculate answer correctness and explanation similarity aggregates."""
    if not records:
        return {
            "total": 0,
            "avg_answer_similarity": 0.0,
            "answer_correct_count": 0,
            "expl_token_f1": 0.0,
            "expl_rouge_l": 0.0,
            "expl_semantic_similarity": 0.0,
            "expl_correct_count": 0,
        }

    total = len(records)
    avg_answer_similarity = sum(r["answer_similarity"] for r in records) / total
    answer_correct_count = sum(r["answer_is_correct"] for r in records)
    expl_token_f1 = sum(r["expl_token_f1"] for r in records) / total
    expl_rouge_l = sum(r["expl_rouge_l"] for r in records) / total
    expl_semantic_similarity = sum(r["expl_semantic_similarity"] for r in records) / total
    expl_correct_count = sum(r["expl_is_correct"] for r in records)

    return {
        "total": total,
        "avg_answer_similarity": avg_answer_similarity,
        "answer_correct_count": answer_correct_count,
        "expl_token_f1": expl_token_f1,
        "expl_rouge_l": expl_rouge_l,
        "expl_semantic_similarity": expl_semantic_similarity,
        "expl_correct_count": expl_correct_count,
    }


def metrics_by_category(records: list) -> dict:
    """Group records by sa_type and compute metrics for each category."""
    category_records = defaultdict(list)
    for rec in records:
        sa_type = rec.get("sa_type", "unknown")
        category_records[sa_type].append(rec)

    category_metrics = {}
    for sa_type, cat_records in category_records.items():
        category_metrics[sa_type] = summary_from_records(cat_records)

    return category_metrics

In [41]:
OUTPUT_DIR = Path("LLM_SA_Comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

out_csv_consolidated = OUTPUT_DIR / f"Comparison_{BASE_TAG}_vs_{COMPARE_TAG}_consolidated.csv"

print(f"[DEBUG] Current working directory: {Path.cwd()}")
print(f"[DEBUG] Base model: {BASE_MODEL} ({BASE_TAG})")
print(f"[DEBUG] Compare model: {COMPARE_MODEL} ({COMPARE_TAG})")
print(f"[DEBUG] Consolidated output CSV: {out_csv_consolidated}")
print(f"[DEBUG] Answer correctness threshold: {ANSWER_SIM_THRESHOLD:.2f}")
print(f"[DEBUG] Explanation correctness threshold: {EXPLANATION_SIM_THRESHOLD:.2f}")

header = [
    "Attack",
    "Total Questions",
    "Answer Similarity (%)",
    "Answer Correct Count (>=90%)",
    "Explanation F1",
    "Explanation ROUGE-L",
    "Explanation Similarity (%)",
    "Explanation Correct Count (>=80%)",
]

# Define category mapping: Category name -> list of SA types
CATEGORY_MAPPING = {
    "Category 1: Anomalous Behavior Identification and Localization": [
        "attack_type", "attack_pattern", "evidence_pattern"
    ],
    "Category 2: CAN Message Identity and Role Structure": [
        "out_of_range_id_count", "dominant_id", "missing_count"
    ],
    "Category 3: Traffic Volume and Distribution Characteristics": [
        "attack_ratio", "id_concentration", "traffic_shift", "fps"
    ],
    "Category 4: Temporal Behavior of CAN Communication": [
        "timestamp_cause", "timing_cause", "max_gap"
    ],
    "Category 5: Payload Value Patterns and Regularities": [
        "payload_behavior", "dominant_var"
    ],
    "Category 6: Frame Format and Protocol-Level Integrity": [
        "dlc_behavior", "dlc_mode", "duplicate_count"
    ],
    "Category 7: Expected and Safety-Critical Message Behavior": [
        "critical_behavior", "missing_count"
    ],
    "Category 8: Payload Dynamics and Baseline Plausibility": [
        "plausibility_relation", "payload_behavior"
    ],
    "Category 9: Data Consistency and Logging Artifacts": [
        "duplicate_count", "timestamp_cause"
    ],
    "Category 10: Attack Interpretation and Decision Signals": [
        "attack_type", "attack_pattern", "evidence_pattern", 
        "attack_ratio", "critical_behavior", "out_of_range_id_count"
    ],
}

dataset_records = {}
all_records = []

for name in SA_DATASET_NAMES:
    base_path = sa_explanations_path(name, BASE_TAG)
    if not base_path.exists():
        base_path = Path(f"{name}_sa_qa/llm_answers/{name.lower()}_sa_explanations_{BASE_TAG}.jsonl")

    compare_path = sa_answers_path(name, COMPARE_TAG)
    if not compare_path.exists():
        compare_path = sa_explanations_path(name, COMPARE_TAG)

    print(f"[DEBUG] Processing {name}:")
    print(f"  Base explanations path: {base_path} -> exists: {base_path.exists()}")
    print(f"  Compare records path: {compare_path} -> exists: {compare_path.exists()}")

    base_records = load_sa_answers(base_path)
    compare_records = load_sa_answers(compare_path)

    print(f"  Base explanations loaded: {len(base_records)}")
    print(f"  Compare records loaded: {len(compare_records)}")

    if not base_records:
        print(f"[WARN] Skip {name}: base explanations missing at {base_path}.")
        continue
    if not compare_records:
        print(f"[WARN] Skip {name}: compare records missing at {compare_path}.")
        continue

    print(f"[INFO] Comparing {name} ({BASE_TAG} vs {COMPARE_TAG})...")
    try:
        eval_records = compute_comparison_records(base_records, compare_records)
        print(f"  Comparison records created: {len(eval_records)}")

        if not eval_records:
            print(f"[ERROR] No comparison records generated for {name}")
            continue

        dataset_records[name] = eval_records
        all_records.extend(eval_records)
    except Exception as e:
        print(f"[ERROR] {name}: {e}")
        import traceback
        traceback.print_exc()
        continue


def build_row(label: str, records: list) -> list:
    metrics = summary_from_records(records)
    return [
        label,
        metrics["total"],
        f"{metrics['avg_answer_similarity'] * 100:.2f}",
        metrics["answer_correct_count"],
        f"{metrics['expl_token_f1']:.3f}",
        f"{metrics['expl_rouge_l']:.3f}",
        f"{metrics['expl_semantic_similarity'] * 100:.2f}",
        metrics["expl_correct_count"],
    ]


# Write all tables to a single consolidated CSV file with section labels
with out_csv_consolidated.open("w", encoding="utf-8", newline="") as f_out:
    writer = csv.writer(f_out)
    
    # Section 1: Overall comparison table (all records combined, no repeats)
    writer.writerow(["=== OVERALL COMPARISON (All SA Types) ==="])
    writer.writerow([])  # Blank line
    writer.writerow(header)
    
    for name in SA_DATASET_NAMES:
        records = dataset_records.get(name, [])
        if not records:
            continue
        writer.writerow(build_row(name, records))
    
    if all_records:
        writer.writerow(build_row("Combined", all_records))
    
    # Section 2+: One table per category
    for category_name, sa_types_in_category in CATEGORY_MAPPING.items():
        writer.writerow([])  # Blank line before new section
        writer.writerow([])  # Extra blank line for clarity
        writer.writerow([f"=== {category_name} ==="])
        writer.writerow([])  # Blank line
        writer.writerow(header)
        
        # Collect all records that match any SA type in this category
        combined_category_records = []
        for name in SA_DATASET_NAMES:
            category_records = [
                r for r in dataset_records.get(name, [])
                if r.get("sa_type", "unknown") in sa_types_in_category
            ]
            if not category_records:
                continue
            writer.writerow(build_row(name, category_records))
            combined_category_records.extend(category_records)
        
        if combined_category_records:
            writer.writerow(build_row("Combined", combined_category_records))

print(f"[INFO] Consolidated comparison table saved to {out_csv_consolidated}")

[DEBUG] Current working directory: c:\Users\Dub\Documents\GitHub\CAN_QA\QA\Create_QA_ShortAnswer_Dataset
[DEBUG] Base model: gemma2:9b (gemma2_9b)
[DEBUG] Compare model: internlm2:latest (internlm2_latest)
[DEBUG] Consolidated output CSV: LLM_SA_Comparison\Comparison_gemma2_9b_vs_internlm2_latest_consolidated.csv
[DEBUG] Answer correctness threshold: 0.90
[DEBUG] Explanation correctness threshold: 0.80
[DEBUG] Processing DoS:
  Base explanations path: DoS_sa_qa\llm_explanations\dos_sa_explanations_gemma2_9b.jsonl -> exists: True
  Compare records path: DoS_sa_qa\llm_answers\dos_sa_answers_internlm2_latest.jsonl -> exists: True
  Base explanations loaded: 916
  Compare records loaded: 916
[INFO] Comparing DoS (gemma2_9b vs internlm2_latest)...
  Comparison records created: 916
[DEBUG] Processing Fuzzy:
  Base explanations path: Fuzzy_sa_qa\llm_explanations\fuzzy_sa_explanations_gemma2_9b.jsonl -> exists: True
  Compare records path: Fuzzy_sa_qa\llm_answers\fuzzy_sa_answers_internlm2_lat

In [ ]:
# # One-question metric check (llama vs gemma baseline)
# llama_rec = {
#     "qa_id": "DoS_SA_000000_00",
#     "dataset": "DoS",
#     "sa_type": "timestamp_cause",
#     "model": "llama3.1:8b",
#     "llm_answer": "Clock drift or desynchronization",
#     "llm_explanation": "The timestamps show irregular and repeating patterns, suggesting clock drift.",
#     "ground_truth": "Normal",
# }

# gemma_rec = {
#     "qa_id": "DoS_SA_000000_00",
#     "dataset": "DoS",
#     "sa_type": "timestamp_cause",
#     "model": "gemma2:9b",
#     "llm_answer": "Timestamps are sequential and consistent.",
#     "ground_truth": "Normal",
# }

# answer_sim = compute_semantic_similarity(llama_rec["llm_answer"], llama_rec["ground_truth"])
# answer_is_correct = answer_sim >= ANSWER_SIM_THRESHOLD

# expl_pred = llama_rec["llm_explanation"]
# expl_gold = gemma_rec["llm_answer"]

# expl_pred_norm = normalize_text(expl_pred)
# expl_gold_norm = normalize_text(expl_gold)

# expl_f1 = compute_token_f1(expl_pred_norm, expl_gold_norm)
# expl_rouge_l = compute_rouge_l(expl_pred_norm, expl_gold_norm)
# expl_sem = compute_semantic_similarity(expl_pred, expl_gold)
# expl_is_correct = expl_sem >= EXPLANATION_SIM_THRESHOLD

# print({
#     "qa_id": llama_rec["qa_id"],
#     "answer_similarity": answer_sim,
#     "answer_similarity_percent": round(answer_sim * 100, 2),
#     "answer_correct_ge_90": answer_is_correct,
#     "explanation_token_f1": round(expl_f1, 6),
#     "explanation_rouge_l": round(expl_rouge_l, 6),
#     "explanation_semantic_similarity": expl_sem,
#     "explanation_similarity_percent": round(expl_sem * 100, 2),
#     "explanation_correct_ge_80": expl_is_correct,
# })

{'qa_id': 'DoS_SA_000000_00', 'answer_similarity': 0.061147335916757584, 'answer_similarity_percent': 6.11, 'answer_correct_ge_90': False, 'explanation_token_f1': 0.266667, 'explanation_rouge_l': 0.266667, 'explanation_semantic_similarity': 0.6730855703353882, 'explanation_similarity_percent': 67.31, 'explanation_correct_ge_80': False}
